In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
procurement = pd.read_csv("../datasets/procurement.csv")
print(procurement.shape)
procurement.head()

(278, 9)


,procurement_id,employee_id,vendor_id,price_system,price_actual,price_difference,account_mismatch_flag,fraud_flag,fraud_probability
0,P4000,EMP1133,V206,6787749,6787749,0,0,0,NaN
1,P4001,EMP1009,V202,2661793,156344,2505449,1,1,NaN
2,P4002,EMP1004,V209,7576651,7576651,0,0,0,NaN
3,P4003,EMP1137,V209,7867218,7867218,0,0,0,NaN
4,P4004,EMP1129,V218,8466027,8466027,0,0,0,NaN


In [3]:
procurement.columns

Index(['procurement_id', 'employee_id', 'vendor_id', 'price_system',
       'price_actual', 'price_difference', 'account_mismatch_flag',
       'fraud_flag', 'fraud_probability'],
      dtype='str')

In [4]:
procurement.info()

<class 'pandas.DataFrame'>
RangeIndex: 278 entries, 0 to 277
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   procurement_id         278 non-null    str    
 1   employee_id            278 non-null    str    
 2   vendor_id              278 non-null    str    
 3   price_system           278 non-null    int64  
 4   price_actual           278 non-null    int64  
 5   price_difference       278 non-null    int64  
 6   account_mismatch_flag  278 non-null    int64  
 7   fraud_flag             278 non-null    int64  
 8   fraud_probability      3 non-null      float64
dtypes: float64(1), int64(5), str(3)
memory usage: 19.7 KB


In [5]:
procurement.isnull().sum()

procurement_id             0
employee_id                0
vendor_id                  0
price_system               0
price_actual               0
price_difference           0
account_mismatch_flag      0
fraud_flag                 0
fraud_probability        275
dtype: int64

In [6]:
procurement = procurement.drop(
    columns=["fraud_probability"]
)

procurement.columns

Index(['procurement_id', 'employee_id', 'vendor_id', 'price_system',
       'price_actual', 'price_difference', 'account_mismatch_flag',
       'fraud_flag'],
      dtype='str')

In [7]:
procurement = procurement.drop(
    columns=[
        "procurement_id",
        "employee_id",
        "vendor_id"
    ]
)

procurement.head()

,price_system,price_actual,price_difference,account_mismatch_flag,fraud_flag
0,6787749,6787749,0,0,0
1,2661793,156344,2505449,1,1
2,7576651,7576651,0,0,0
3,7867218,7867218,0,0,0
4,8466027,8466027,0,0,0


In [8]:
procurement["fraud_flag"].value_counts()

fraud_flag
0    191
1     87
Name: count, dtype: int64

In [9]:
X = procurement.drop("fraud_flag", axis=1)

y = procurement["fraud_flag"]

print(X.head())
print(y.head())

   price_system  price_actual  price_difference  account_mismatch_flag
0       6787749       6787749                 0                      0
1       2661793        156344           2505449                      1
2       7576651       7576651                 0                      0
3       7867218       7867218                 0                      0
4       8466027       8466027                 0                      0
0    0
1    1
2    0
3    0
4    0
Name: fraud_flag, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (222, 4)
X_test : (56, 4)
y_train: (222,)
y_test : (56,)


In [11]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [12]:
predictions = model.predict(X_test)

print(predictions[:10])

[1 1 0 1 1 0 1 1 0 0]


In [13]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.9285714285714286


In [14]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, predictions))

[[30  3]
 [ 1 22]]


In [15]:
from sklearn.metrics import classification_report

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.97      0.91      0.94        33
           1       0.88      0.96      0.92        23

    accuracy                           0.93        56
   macro avg       0.92      0.93      0.93        56
weighted avg       0.93      0.93      0.93        56



In [16]:
import pickle
with open("../models/procurement_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved successfully!")

Model saved successfully!


In [17]:
# Load the saved model
with open("../models/procurement_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

# Test predictions
predictions = loaded_model.predict(X_test[:5])

# Test probability predictions
probabilities = loaded_model.predict_proba(X_test[:5])[:, 1]

print("Predictions:")
print(predictions)

print("\nFraud Probabilities:")
print(probabilities)

Predictions:
[1 1 0 1 1]

Fraud Probabilities:
[0.85 0.96 0.   1.   0.95]
